# Rodent vs. No-Rodent Classifier — MobileNetV2 (Colab)

Fine-tunes an ImageNet-pretrained MobileNetV2 on your rodent / no-rodent dataset,
then exports a quantized int8 TFLite model for edge deployment on ESP32-CAM.

MobileNetV2 uses ReLU6 activations throughout (hard ceiling at 6) with no
hard-swish or squeeze-excite blocks, making it the most int8-PTQ-friendly
MobileNet variant and the reference model in the TFLite quantization docs.

**Before running:** set `DATASET_ROOT` in the "Configure paths" cell below.


## 1. Setup

In [2]:
!pip install -q scikit-learn ai-edge-litert


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 99.3 MB/s eta 0:00:00


In [2]:
import os
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print("TF version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


TF version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Mount Drive and configure paths

In [5]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
# ---- EDIT THIS ----
DATASET_ROOT = "/content/ei_curated"   # folder containing training/ and testing/
# --------------------

TRAIN_DIR = os.path.join(DATASET_ROOT, "training")
TEST_DIR  = os.path.join(DATASET_ROOT, "testing")

OUTPUT_DIR = "/content/drive/MyDrive/rodent_model_output_mobilenetv2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE   = 224        # MobileNetV2 native input size
BATCH_SIZE = 32
SEED       = 42


## 3. Build a file list from the Edge Impulse export

Each of `training/` and `testing/` is a flat folder of images plus one `bounding_boxes.labels`
JSON file (the Edge Impulse bounding-box label format). An image counts as `rodent` if it has
at least one bounding box listed; otherwise (missing from the file, or an empty box list --
which includes your generated background images) it's `no_rodent`.

No grouping/leakage handling here, since the dataset is already cleaned and reduced.

In [5]:
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

def parse_ei_labels(folder):
    """Reads bounding_boxes.labels from an Edge Impulse export folder and returns
    {filename: has_rodent_bool}."""
    labels_path = os.path.join(folder, "bounding_boxes.labels")
    with open(labels_path, "r") as f:
        data = json.load(f)

    boxes_by_file = data.get("boundingBoxes", data)  # tolerate slightly different EI export shapes
    has_rodent = {}
    for fname, boxes in boxes_by_file.items():
        has_rodent[fname] = bool(boxes) and len(boxes) > 0
    return has_rodent

def collect_files(folder):
    folder = Path(folder)
    has_rodent = parse_ei_labels(folder)

    rows = []
    for p in folder.iterdir():
        if p.suffix.lower() not in VALID_EXTS:
            continue
        is_rodent = has_rodent.get(p.name, False)
        rows.append({
            "filepath": str(p),
            "label": "rodent" if is_rodent else "no_rodent",
        })
    return pd.DataFrame(rows)

df_train_full = collect_files(TRAIN_DIR)
df_test = collect_files(TEST_DIR)

df_train_full["label_idx"] = (df_train_full["label"] == "rodent").astype(int)
df_test["label_idx"] = (df_test["label"] == "rodent").astype(int)

print("Training folder:")
print(df_train_full["label"].value_counts())
print("\nTesting folder:")
print(df_test["label"].value_counts())


Training folder:
label
rodent       3600
no_rodent     480
Name: count, dtype: int64

Testing folder:
label
rodent       900
no_rodent    120
Name: count, dtype: int64


## 4. Train / val split (testing/ is already your held-out test set)

A plain stratified random split of training/ into train and val -- no grouping, since the
dataset is already cleaned.

In [6]:
df_train, df_val = train_test_split(
    df_train_full,
    test_size=0.15,
    stratify=df_train_full["label_idx"],
    random_state=SEED,
)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

print(f"Train: {len(df_train)} images")
print(f"Val:   {len(df_val)} images")
print(f"Test:  {len(df_test)} images")
print("\nClass balance (train):")
print(df_train['label'].value_counts(normalize=True))


Train: 3468 images
Val:   612 images
Test:  1020 images

Class balance (train):
label
rodent       0.882353
no_rodent    0.117647
Name: proportion, dtype: float64


## 5. Build tf.data pipelines — with ESP32-CAM camera simulation

The training augmentation now aggressively mimics the OV2640 sensor on the
ESP32-CAM, closing the domain gap that made the deployed model score ~0.03
on frames it should recognise. On a random ~70% of training images we apply:

downscale-upscale (sensor softness/blur), JPEG compression artifacts
(quality 10–50), sensor noise, and — on all images — stronger
brightness/contrast/saturation/hue jitter to cover the camera's murky,
colour-cast output.

Augmentation runs in [0,1] pixel space (JPEG/blur ops need real pixels);
ImageNet normalisation is applied afterwards, for all splits. Validation and
test stay clean, so reported metrics remain comparable to previous runs.


In [7]:
AUTOTUNE = tf.data.AUTOTUNE

# MobileNetV2 was trained with ImageNet mean/std normalisation.
# Normalisation is applied as the LAST pipeline step (after augmentation),
# because the camera-simulation ops (JPEG, blur, noise) need [0,1] pixels.
IMAGENET_MEAN = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
IMAGENET_STD  = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)

def load_image_raw(filepath, label):
    """Load and resize to [0,1] float — WITHOUT normalisation."""
    img = tf.io.read_file(filepath)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.ensure_shape(img, [IMG_SIZE, IMG_SIZE, 3])
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

def normalize(img, label):
    """ImageNet normalisation — applied to ALL splits as the final step."""
    return (img - IMAGENET_MEAN) / IMAGENET_STD, label

# ── ESP32-CAM (OV2640) camera simulation ──────────────────────────────────────

def _camera_degrade(img):
    """Make a clean photo look like it came from the ESP32-CAM:
    resolution loss + JPEG artifacts + sensor noise."""
    # 1. Resolution loss / blur: downscale to a random small size, upscale back.
    #    (The device captures QVGA and nearest-neighbor resizes to 224 — detail
    #    is gone. Bilinear down-up reproduces that softness.)
    small = tf.random.uniform([], 96, 200, dtype=tf.int32)
    img = tf.image.resize(img, [small, small])
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.clip_by_value(img, 0.0, 1.0)

    # 2. JPEG compression artifacts (device captures JPEG at quality ~12).
    img = tf.image.random_jpeg_quality(img, 10, 50)

    # 3. Sensor noise.
    noise_std = tf.random.uniform([], 0.01, 0.05)
    img = img + tf.random.normal(tf.shape(img), stddev=noise_std)
    return tf.clip_by_value(img, 0.0, 1.0)

def augment(img, label):
    img = tf.image.random_flip_left_right(img)

    # Photometric jitter (always) — the OV2640's murky, low-contrast,
    # colour-cast output. Ranges are deliberately wide.
    img = tf.image.random_brightness(img, max_delta=0.20)
    img = tf.image.random_contrast(img, lower=0.5, upper=1.3)
    img = tf.image.random_saturation(img, lower=0.4, upper=1.2)
    img = tf.image.random_hue(img, max_delta=0.05)
    img = tf.clip_by_value(img, 0.0, 1.0)

    # Heavy camera-quality degradation on ~70% of images. The other 30%
    # stay clean so the model keeps its accuracy on good imagery too.
    img = tf.cond(tf.random.uniform([]) < 0.7,
                  lambda: _camera_degrade(img),
                  lambda: img)
    img = tf.ensure_shape(img, [IMG_SIZE, IMG_SIZE, 3])
    return img, label

def make_dataset(df_split, training=False):
    ds = tf.data.Dataset.from_tensor_slices((df_split["filepath"].values,
                                              df_split["label_idx"].values.astype(np.float32)))
    ds = ds.map(load_image_raw, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(1000, seed=SEED)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.map(normalize, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(df_train, training=True)
val_ds   = make_dataset(df_val,   training=False)
test_ds  = make_dataset(df_test,  training=False)

# ── Sanity preview: what the model will see (before normalisation) ────────────
import matplotlib.pyplot as plt
_preview = tf.data.Dataset.from_tensor_slices(
    (df_train["filepath"].values[:8], df_train["label_idx"].values[:8].astype(np.float32)))
_preview = _preview.map(load_image_raw).map(augment)
plt.figure(figsize=(14, 4))
for i, (im, lb) in enumerate(_preview):
    plt.subplot(1, 8, i + 1)
    plt.imshow(im.numpy())
    plt.title('rodent' if lb.numpy() == 1 else 'no_rodent', fontsize=8)
    plt.axis('off')
plt.suptitle('Augmented samples — should look like bad ESP32-CAM frames', fontsize=10)
plt.tight_layout()
plt.show()


## 6. Build the model

`keras.applications.MobileNetV2` uses ReLU6 activations throughout — a hard
ceiling at 6 that gives int8 per-tensor quantization a bounded, predictable
activation range. This makes it the reference model in the TFLite quantization
docs and the right choice for int8 PTQ on ESP32-CAM.

Swap `alpha=1.0` (default) for `alpha=0.5` if you need a smaller model at the
cost of some accuracy.


In [8]:
def build_model(trainable_backbone=False):
    # MobileNetV2 does not have include_preprocessing — normalisation is
    # applied in load_image (ImageNet mean/std). ReLU6 throughout means all
    # activations are bounded in [0, 6] — exactly what int8 PTQ needs.
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        alpha=0.35,          # width multiplier — ~400KB int8 model, fits ESP32-CAM flash
        include_top=False,
        weights="imagenet",
        pooling="avg",
    )
    base_model.trainable = trainable_backbone

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        base_model,
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])
    return model

model = build_model(trainable_backbone=False)
model.summary()


2019640/2019640 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_0.35_224            │ (None, 1280)           │       410,208 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 411,489 (1.57 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 410,208 (1.56 MB)

## 7. Phase 1 — train the classification head (backbone frozen)

In [9]:
n_rodent = (df_train["label_idx"] == 1).sum()
n_no_rodent = (df_train["label_idx"] == 0).sum()
total = n_rodent + n_no_rodent
class_weight = {
    0: total / (2 * n_no_rodent),
    1: total / (2 * n_rodent),
}
print("Class weights:", class_weight)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(os.path.join(OUTPUT_DIR, "phase1_best.keras"),
                                        monitor="val_loss", save_best_only=True),
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weight,
    callbacks=callbacks,
)


Class weights: {0: np.float64(4.25), 1: np.float64(0.5666666666666667)}
Epoch 1/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 68s 365ms/step - accuracy: 0.8472 - loss: 0.4202 - precision: 0.9613 - recall: 0.8614 - val_accuracy: 0.9412 - val_loss: 0.2402 - val_precision: 0.9773 - val_recall: 0.9556
Epoch 2/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 18s 118ms/step - accuracy: 0.9360 - loss: 0.2585 - precision: 0.9823 - recall: 0.9444 - val_accuracy: 0.9330 - val_loss: 0.2680 - val_precision: 0.9921 - val_recall: 0.9315
Epoch 3/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 17s 124ms/step - accuracy: 0.9426 - loss: 0.2140 - precision: 0.9848 - recall: 0.9497 - val_accuracy: 0.9542 - val_loss: 0.1883 - val_precision: 0.9923 - val_recall: 0.9556
Epoch 4/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 16s 121ms/step - accuracy: 0.9533 - loss: 0.1894 - precision: 0.9885 - recall: 0.9582 - val_accuracy: 0.9526 - val_loss: 0.1899 - val_precision: 0.9923 - val_recall: 0.9537
Epoch 5/15
109/109 ━━━━━━━━━━━━━━━━━━━━ 16s 121ms/step - accuracy: 0.9619 - 

## 8. (Optional) Phase 2 — fine-tune the backbone

Only run this if phase-1 accuracy/precision/recall (section 9, evaluated on `model`) isn't good
enough. Unfreezes the backbone and continues training with a much lower learning rate. If you
skip this cell, just use `model` (not `model_ft`) in sections 9-10 below.

In [ ]:
model_ft = build_model(trainable_backbone=True)
model_ft.set_weights(model.get_weights())  # carry over phase-1 weights

model_ft.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")]
)

callbacks_ft = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(os.path.join(OUTPUT_DIR, "phase2_best.keras"),
                                        monitor="val_loss", save_best_only=True),
]

history2 = model_ft.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weight,
    callbacks=callbacks_ft,
)


Epoch 1/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 96s 413ms/step - accuracy: 0.4363 - loss: 0.7711 - precision: 0.9973 - recall: 0.3621 - val_accuracy: 0.9706 - val_loss: 0.1023 - val_precision: 0.9798 - val_recall: 0.9870
Epoch 2/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 14s 110ms/step - accuracy: 0.6984 - loss: 0.3823 - precision: 0.9980 - recall: 0.6595 - val_accuracy: 0.9706 - val_loss: 0.0983 - val_precision: 0.9798 - val_recall: 0.9870
Epoch 3/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 21s 111ms/step - accuracy: 0.8362 - loss: 0.2611 - precision: 0.9941 - recall: 0.8193 - val_accuracy: 0.9706 - val_loss: 0.0960 - val_precision: 0.9745 - val_recall: 0.9926
Epoch 4/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 13s 103ms/step - accuracy: 0.9037 - loss: 0.1962 - precision: 0.9953 - recall: 0.8951 - val_accuracy: 0.9706 - val_loss: 0.0940 - val_precision: 0.9745 - val_recall: 0.9926
Epoch 5/10
109/109 ━━━━━━━━━━━━━━━━━━━━ 21s 108ms/step - accuracy: 0.9175 - loss: 0.1764 - precision: 0.9936 - recall: 0.9124 - val_accuracy: 0.9690

## 9. Evaluate on the held-out test set

Set `FINAL_MODEL` below to whichever model you actually want to evaluate/export: `model`
(phase 1 only) or `model_ft` (if you ran phase 2).

In [10]:
FINAL_MODEL = model   # change to model_ft if you ran phase 2

test_loss, test_acc, test_prec, test_rec = FINAL_MODEL.evaluate(test_ds)
print(f"Test accuracy:  {test_acc:.4f}")
print(f"Test precision: {test_prec:.4f}")
print(f"Test recall:    {test_rec:.4f}")

y_true = df_test["label_idx"].values
y_pred_prob = FINAL_MODEL.predict(test_ds).ravel()
y_pred = (y_pred_prob > 0.5).astype(int)

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=["no_rodent", "rodent"]))

print("Confusion matrix (rows=true, cols=pred), order [no_rodent, rodent]:")
print(confusion_matrix(y_true, y_pred))


32/32 ━━━━━━━━━━━━━━━━━━━━ 15s 486ms/step - accuracy: 0.9569 - loss: 0.1338 - precision: 0.9920 - recall: 0.9589
Test accuracy:  0.9569
Test precision: 0.9920
Test recall:    0.9589
32/32 ━━━━━━━━━━━━━━━━━━━━ 13s 281ms/step

Classification report:
              precision    recall  f1-score   support

   no_rodent       0.75      0.94      0.84       120
      rodent       0.99      0.96      0.98       900

    accuracy                           0.96      1020
   macro avg       0.87      0.95      0.91      1020
weighted avg       0.96      0.96      0.96      1020

Confusion matrix (rows=true, cols=pred), order [no_rodent, rodent]:
[[113   7]
 [ 37 863]]


## 10. Convert to TFLite — full integer (int8) quantization

Uses signed int8 I/O — the modern TFLite quantization scheme supported by the
XNNPACK delegate and TFLite Micro. Calibrate on unaugmented training data so
the int8 range reflects real inference-time images, not jittered training crops.


In [11]:
saved_model_dir = os.path.join(OUTPUT_DIR, "saved_model")
FINAL_MODEL.export(saved_model_dir)

# Calibrate on unaugmented data — train_ds has augmentation baked in, which
# skews the int8 calibration range away from real inference-time images.
calib_ds = make_dataset(df_train, training=False)

def representative_dataset():
    for images, _ in calib_ds.take(50):
        for img in images:
            yield [tf.expand_dims(img, 0)]

converter_int8 = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type  = tf.int8
converter_int8.inference_output_type = tf.int8

tflite_int8 = converter_int8.convert()

with open(os.path.join(OUTPUT_DIR, "model_int8.tflite"), "wb") as f:
    f.write(tflite_int8)

print("Saved model_int8.tflite —", len(tflite_int8) / 1024, "KB")


Saved artifact at '/content/drive/MyDrive/rodent_model_output_mobilenetv2/saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_155')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  139425207079888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139425207078736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139425207081232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139425207080464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139425207080080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139425207078544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139425207081424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139425207082192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139425207081808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139425207080656: TensorSpec(sh

## 11. Verify the int8 model on the held-out test set

Uses the newer `ai_edge_litert` interpreter (the `tf.lite.Interpreter` API is being
phased out of TensorFlow starting TF 2.20).

**Threshold note:** MobileNetV2's ReLU6 activations quantize cleanly, so the int8
model should track float accuracy closely and the default threshold of 0.5 should
work. `INT8_THRESHOLD` is left as a tunable constant; the sweep at the end of this
cell will confirm whether any adjustment is needed.


In [12]:
from ai_edge_litert.interpreter import Interpreter, OpResolverType

INT8_THRESHOLD = 0.5

# Must match load_image exactly
_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def evaluate_tflite_int8(tflite_path, df_split, threshold=0.5):
    """Evaluate a fully-integer (signed int8 I/O) TFLite model."""
    interpreter = Interpreter(
        model_path=tflite_path,
        experimental_op_resolver_type=OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES,
    )
    interpreter.allocate_tensors()
    input_details  = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    in_scale,  in_zero_point  = input_details['quantization']
    out_scale, out_zero_point = output_details['quantization']

    y_true, y_pred, raw_probs = [], [], []

    for _, row in df_split.iterrows():
        img = tf.io.read_file(row['filepath'])
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = tf.cast(img, tf.float32) / 255.0
        img = (img - _MEAN) / _STD          # ImageNet normalisation — matches load_image

        # Quantise normalised float to int8 using the tensor's scale/zero-point.
        q = tf.round(img / in_scale) + in_zero_point
        q = tf.clip_by_value(q, -128, 127)
        img = tf.cast(q, tf.int8)
        img = tf.expand_dims(img, 0)

        interpreter.set_tensor(input_details['index'], img)
        interpreter.invoke()
        raw_out = interpreter.get_tensor(output_details['index'])[0][0]

        pred = (float(raw_out) - out_zero_point) * out_scale
        y_true.append(int(row['label_idx']))
        y_pred.append(1 if pred > threshold else 0)
        raw_probs.append(pred)

    acc = np.mean(np.array(y_true) == np.array(y_pred))
    return acc, np.array(y_true), np.array(y_pred), np.array(raw_probs)


acc_int8, yt_int8, yp_int8, probs_int8 = evaluate_tflite_int8(
    os.path.join(OUTPUT_DIR, 'model_int8.tflite'), df_test,
    threshold=INT8_THRESHOLD)

print(f'Int8 TFLite test accuracy (threshold={INT8_THRESHOLD}): {acc_int8:.4f}')
print(classification_report(yt_int8, yp_int8, target_names=['no_rodent', 'rodent']))
print('Confusion matrix (rows=true, cols=pred), order [no_rodent, rodent]:')
print(confusion_matrix(yt_int8, yp_int8))

print('\nInt8 raw prob stats -> min: {:.4f}  max: {:.4f}  mean: {:.4f}'.format(
    probs_int8.min(), probs_int8.max(), probs_int8.mean()))

print('\nThreshold sweep:')
print(f"{'thresh':>8}  {'acc':>6}  {'rodent_recall':>13}  {'no_rodent_recall':>16}")
for thresh in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    preds = (probs_int8 > thresh).astype(int)
    acc_t = np.mean(preds == yt_int8)
    rec_r = preds[yt_int8 == 1].mean()
    rec_n = preds[yt_int8 == 0].mean()
    print(f'{thresh:>8.2f}  {acc_t:>6.4f}  {rec_r:>13.3f}  {rec_n:>16.3f}')


Int8 TFLite test accuracy (threshold=0.5): 0.9559
              precision    recall  f1-score   support

   no_rodent       0.75      0.94      0.83       120
      rodent       0.99      0.96      0.97       900

    accuracy                           0.96      1020
   macro avg       0.87      0.95      0.90      1020
weighted avg       0.96      0.96      0.96      1020

Confusion matrix (rows=true, cols=pred), order [no_rodent, rodent]:
[[113   7]
 [ 38 862]]

Int8 raw prob stats -> min: 0.0000  max: 0.9961  mean: 0.8098

Threshold sweep:
  thresh     acc  rodent_recall  no_rodent_recall
    0.15  0.9618          0.984             0.208
    0.20  0.9667          0.983             0.158
    0.25  0.9657          0.978             0.125
    0.30  0.9637          0.972             0.100
    0.35  0.9637          0.968             0.067
    0.40  0.9598          0.962             0.058
    0.45  0.9578          0.960             0.058
    0.50  0.9559          0.958             0.058


## 12. Save labels file (needed alongside the .tflite model for inference)

In [13]:
labels = ["no_rodent", "rodent"]
with open(os.path.join(OUTPUT_DIR, "labels.txt"), "w") as f:
    f.write("\n".join(labels))

print("All outputs saved to:", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))


All outputs saved to: /content/drive/MyDrive/rodent_model_output_mobilenetv2
['phase1_best.keras', 'saved_model', 'model_int8.tflite', 'labels.txt']


## 13. Package the int8 model as an Arduino library

Converts `model_int8.tflite` into a C byte array and assembles an Arduino library
folder (`library.properties`, `src/`, an example sketch), then zips it — ready
to install via Arduino IDE's *Sketch → Include Library → Add .ZIP Library*.

**One real dependency you still need to install yourself:** the
[TensorFlow Lite for Microcontrollers Arduino library](https://github.com/tensorflow/tflite-micro-arduino-examples)
(search "Arduino_TensorFlowLite" or "Chirale_TensorFlowLite" in the Arduino
Library Manager). Your generated library depends on it at compile time.


In [14]:
LIBRARY_NAME = "RodentDetectorV2"   # change if you want a different library/class name

lib_root    = os.path.join(OUTPUT_DIR, LIBRARY_NAME)
src_dir     = os.path.join(lib_root, "src")
example_dir = os.path.join(lib_root, "examples", f"{LIBRARY_NAME}_Inference")

os.makedirs(src_dir,     exist_ok=True)
os.makedirs(example_dir, exist_ok=True)


### 13a. Convert the .tflite file into a C byte array

In [15]:
def tflite_to_c_array(tflite_path, var_name="g_model"):
    with open(tflite_path, "rb") as f:
        data = f.read()

    lines = []
    for i in range(0, len(data), 12):
        chunk = data[i:i + 12]
        lines.append(", ".join(f"0x{b:02x}" for b in chunk))
    body = ",\n  ".join(lines)

    header = f"""// Auto-generated from {os.path.basename(tflite_path)} -- do not edit by hand.
#ifndef {var_name.upper()}_DATA_H
#define {var_name.upper()}_DATA_H

alignas(8) const unsigned char {var_name}[] = {{
  {body}
}};
const unsigned int {var_name}_len = {len(data)};

#endif  // {var_name.upper()}_DATA_H
"""
    return header, len(data)

model_header, model_size = tflite_to_c_array(
    os.path.join(OUTPUT_DIR, "model_int8.tflite"), var_name="g_model"
)

with open(os.path.join(src_dir, "model_data.h"), "w") as f:
    f.write(model_header)

print(f"Embedded model: {model_size} bytes ({model_size/1024:.1f} KB) -> src/model_data.h")


Embedded model: 621720 bytes (607.1 KB) -> src/model_data.h


### 13b. Write library.properties

In [16]:
library_properties = f"""name={LIBRARY_NAME}
version=1.0.0
author=Your Name
maintainer=Your Name <you@example.com>
sentence=On-device rodent / no-rodent classification using a quantized MobileNetV2 model.
paragraph=Runs TensorFlow Lite for Microcontrollers inference on a quantized int8 MobileNetV2 model to classify camera frames as rodent or no_rodent.
category=Sensors
url=
architectures=*
depends=Arduino_TensorFlowLite
"""

with open(os.path.join(lib_root, "library.properties"), "w") as f:
    f.write(library_properties)


### 13c. Write the library header/source

A thin wrapper class around the TFLite Micro interpreter. It expects a `{IMG_SIZE}x{IMG_SIZE}x3`
uint8 image buffer as input and returns a float probability (rodent likelihood) as output.

In [17]:
header_code = f"""#ifndef {LIBRARY_NAME.upper()}_H
#define {LIBRARY_NAME.upper()}_H

#include <TensorFlowLite.h>

class {LIBRARY_NAME} {{
  public:
    {LIBRARY_NAME}();
    bool begin();
    // input: pointer to {IMG_SIZE}*{IMG_SIZE}*3 uint8 RGB bytes, already resized to model input size
    // returns: probability that the image contains a rodent (0.0 - 1.0)
    float predict(const uint8_t* image_data);

  private:
    bool _initialized = false;
}};

#endif  // {LIBRARY_NAME.upper()}_H
"""

with open(os.path.join(src_dir, f"{LIBRARY_NAME}.h"), "w") as f:
    f.write(header_code)


In [18]:
cpp_code = f"""#include "{LIBRARY_NAME}.h"
#include "model_data.h"

#include <tensorflow/lite/micro/micro_interpreter.h>
#include <tensorflow/lite/micro/micro_mutable_op_resolver.h>
#include <tensorflow/lite/micro/micro_log.h>
#include <tensorflow/lite/schema/schema_generated.h>

namespace {{
  constexpr int kTensorArenaSize = 200 * 1024;  // tune down/up depending on your board's RAM
  alignas(16) uint8_t tensor_arena[kTensorArenaSize];

  const tflite::Model* model = nullptr;
  tflite::MicroInterpreter* interpreter = nullptr;
  TfLiteTensor* input = nullptr;
  TfLiteTensor* output = nullptr;
}}

{LIBRARY_NAME}::{LIBRARY_NAME}() {{}}

bool {LIBRARY_NAME}::begin() {{
  model = tflite::GetModel(g_model);
  if (model->version() != TFLITE_SCHEMA_VERSION) {{
    MicroPrintf("Model schema version mismatch!");
    return false;
  }}

  // Add only the ops your model actually needs (keeps flash usage down).
  // If you get an "unresolved op" error at runtime, add the missing op here.
  static tflite::MicroMutableOpResolver<10> resolver;
  resolver.AddConv2D();
  resolver.AddDepthwiseConv2D();
  resolver.AddFullyConnected();
  resolver.AddLogistic();
  resolver.AddAveragePool2D();
  resolver.AddReshape();
  resolver.AddAdd();
  resolver.AddMul();
  resolver.AddPad();
  resolver.AddMean();

  static tflite::MicroInterpreter static_interpreter(
      model, resolver, tensor_arena, kTensorArenaSize);
  interpreter = &static_interpreter;

  if (interpreter->AllocateTensors() != kTfLiteOk) {{
    MicroPrintf("AllocateTensors() failed");
    return false;
  }}

  input = interpreter->input(0);
  output = interpreter->output(0);
  _initialized = true;
  return true;
}}

float {LIBRARY_NAME}::predict(const uint8_t* image_data) {{
  if (!_initialized) return -1.0f;

  memcpy(input->data.uint8, image_data, input->bytes);

  if (interpreter->Invoke() != kTfLiteOk) {{
    MicroPrintf("Invoke() failed");
    return -1.0f;
  }}

  uint8_t raw = output->data.uint8[0];
  float scale = output->params.scale;
  int zero_point = output->params.zero_point;
  return (raw - zero_point) * scale;
}}
"""

with open(os.path.join(src_dir, f"{LIBRARY_NAME}.cpp"), "w") as f:
    f.write(cpp_code)


### 13d. Write an example sketch

In [19]:
example_sketch = f"""#include <{LIBRARY_NAME}.h>

{LIBRARY_NAME} detector;

uint8_t image_buffer[{IMG_SIZE} * {IMG_SIZE} * 3];

void setup() {{
  Serial.begin(115200);
  while (!Serial) {{}}

  if (!detector.begin()) {{
    Serial.println("Failed to initialize model.");
    while (1) {{}}
  }}
  Serial.println("Model ready.");
}}

void loop() {{
  // TODO: capture a frame into image_buffer here, resized to {IMG_SIZE}x{IMG_SIZE}
  float rodent_prob = detector.predict(image_buffer);
  Serial.print("Rodent probability: ");
  Serial.println(rodent_prob);

  delay(1000);
}}
"""

with open(os.path.join(example_dir, f"{LIBRARY_NAME}_Inference.ino"), "w") as f:
    f.write(example_sketch)


### 13e. Zip the library

In [20]:
zip_path_base = os.path.join(OUTPUT_DIR, LIBRARY_NAME)
zip_path = shutil.make_archive(zip_path_base, "zip", root_dir=OUTPUT_DIR, base_dir=LIBRARY_NAME)

print("Arduino library zipped at:", zip_path)
print("\nInstall via Arduino IDE: Sketch -> Include Library -> Add .ZIP Library... -> select this file.")
print("Remember to also install the 'Arduino_TensorFlowLite' (or 'Chirale_TensorFlowLite') library")
print("from the Library Manager first -- this generated library depends on it.")


Arduino library zipped at: /content/drive/MyDrive/rodent_model_output_mobilenetv2/RodentDetectorV2.zip

Install via Arduino IDE: Sketch -> Include Library -> Add .ZIP Library... -> select this file.
Remember to also install the 'Arduino_TensorFlowLite' (or 'Chirale_TensorFlowLite') library
from the Library Manager first -- this generated library depends on it.


In [6]:
import numpy as np, tensorflow as tf, os
from ai_edge_litert.interpreter import Interpreter, OpResolverType
OUTPUT_DIR = "/content/drive/MyDrive/rodent_model_output_mobilenetv2"
# Same synthetic pattern as the sketch
img = np.zeros((224, 224, 3), dtype=np.uint8)
for y in range(224):
    for x in range(224):
        v = (x + y) & 0xFF
        img[y, x] = [v, v ^ 0x55, 255 - v]

_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

interp = Interpreter(
    model_path=os.path.join(OUTPUT_DIR, 'model_int8.tflite'),
    experimental_op_resolver_type=OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES)
interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
in_s, in_z = inp['quantization']; out_s, out_z = out['quantization']

norm = (img.astype(np.float32) / 255.0 - _MEAN) / _STD
q = np.clip(np.round(norm / in_s) + in_z, -128, 127).astype(np.int8)
interp.set_tensor(inp['index'], q[None, ...])
interp.invoke()
raw = interp.get_tensor(out['index'])[0][0]
print("Colab self-test probability:", (float(raw) - out_z) * out_s)

Colab self-test probability: 0.0


In [7]:
from PIL import Image
import numpy as np, os
from ai_edge_litert.interpreter import Interpreter, OpResolverType

img = np.array(Image.open('/content/test.webp').convert('RGB'))
print("shape:", img.shape)   # should be (224, 224, 3)

_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

interp = Interpreter(
    model_path=os.path.join(OUTPUT_DIR, 'model_int8.tflite'),
    experimental_op_resolver_type=OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES)
interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
in_s, in_z = inp['quantization']; out_s, out_z = out['quantization']

def predict(arr):
    norm = (arr.astype(np.float32) / 255.0 - _MEAN) / _STD
    q = np.clip(np.round(norm / in_s) + in_z, -128, 127).astype(np.int8)
    interp.set_tensor(inp['index'], q[None, ...])
    interp.invoke()
    raw = interp.get_tensor(out['index'])[0][0]
    return (float(raw) - out_z) * out_s

print("As-is (RGB):        ", predict(img))
print("Channels reversed:  ", predict(img[..., ::-1]))

shape: (224, 224, 3)
As-is (RGB):         0.03515625
Channels reversed:   0.01953125
